# Day 60 — Performance & Memory Optimization
**Month 3 | Week 6 | dtype reduction · vectorization · query/eval · chunked processing**

---

> **Real-world framing:**
>
> A Python script that works on 500 rows will often crash or freeze on 500,000.
> This day closes that gap.
>
> Memory optimization: load a 2 GB file without running out of RAM by choosing smarter dtypes.
> Vectorization: make a revenue calculation 36× faster by eliminating row-by-row loops.
> query() / eval(): write filter logic that reads like SQL and scales to production data.
> Chunking: process files that don't fit in memory by reading them 100 rows at a time.
>
> These are the four techniques every data engineer asks about in interviews —
> and every analyst needs the moment a client's dataset grows beyond toy size.

---
**Score: 80 pts + 10★ Bonus**


## Section 1 — Raw Data
*Never modify this cell. All tasks work on `df` which is a copy.*

In [2]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# ── Dataset: 500 ShopEase orders (seed=60) ─────────────────────────────────
rng = np.random.default_rng(seed=60)
n   = 500

raw = pd.DataFrame({
    'order_id'    : [f'ORD{4000+i}' for i in range(n)],
    'customer_id' : rng.integers(1001, 1061, n).astype('int64'),
    'region'      : rng.choice(['North','South','East','West'], n),
    'category'    : rng.choice(['Electronics','Clothing','Home','Sports','Books'], n),
    'segment'     : rng.choice(['Regular','Premium','VIP'], n),
    'status'      : rng.choice(['Delivered','Returned','Pending'], n, p=[0.70,0.20,0.10]),
    'units'       : rng.integers(1, 25, n).astype('int64'),
    'unit_price'  : rng.uniform(50, 600, n).astype('float64'),
    'discount_pct': rng.choice([0,5,10,15,20], n).astype('int64'),
    'return_flag' : rng.choice([0, 1], n, p=[0.80, 0.20]).astype('int64'),
})
raw['revenue'] = (raw['units'] * raw['unit_price'] * (1 - raw['discount_pct']/100)).round(2)

# ── Working copy (never touch `raw`) ───────────────────────────────────────
df = raw.copy()

print("Dataset shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 3 rows:")
df.head(3)


Dataset shape: (500, 11)

Column dtypes:
order_id         object
customer_id       int64
region           object
category         object
segment          object
status           object
units             int64
unit_price      float64
discount_pct      int64
return_flag       int64
revenue         float64
dtype: object

First 3 rows:


,order_id,customer_id,region,category,segment,status,units,unit_price,discount_pct,return_flag,revenue
0,ORD4000,1011,West,Sports,VIP,Delivered,16,487.339210,15,1,6627.81
1,ORD4001,1020,West,Clothing,VIP,Delivered,22,439.692154,10,1,8705.90
2,ORD4002,1024,North,Electronics,Premium,Delivered,2,366.615459,20,0,586.58


## Section 2 — Concept Notes

---

### 1. Memory Optimization — Why It Matters

When Pandas loads a CSV it makes conservative guesses:
- Any integer column → `int64` (8 bytes per value)
- Any float column → `float64` (8 bytes per value)
- Any text column → `object` (~50+ bytes per value — it's a Python string object)

For a 500-row toy dataset this doesn't matter. For 5 million rows, `object` columns alone
can eat 250 MB of RAM. The fix is downcast to the smallest correct dtype.

**The dtype hierarchy (integers):**
| dtype | bits | value range | bytes/value |
|-------|------|-------------|-------------|
| int8  |  8   | -128 to 127 | 1 |
| int16 | 16   | -32,768 to 32,767 | 2 |
| int32 | 32   | ±2.1 billion | 4 |
| int64 | 64   | ±9.2 × 10¹⁸ | 8 |

**Rule of thumb:** choose the smallest dtype whose range covers ALL values in the column.
`discount_pct` only goes 0–20 → `int8` is safe. `customer_id` goes up to 1060 → `int16` works.

**`category` dtype** — the biggest single win.
A column like `region` with only 4 unique values stored as `object` takes ~26 KB for 500 rows
(one Python string object per row). As `category`, Pandas stores an integer lookup table — 0.7 KB.
That's a 97% reduction for that column alone.

```python
# Measure memory before
df.memory_usage(deep=True).sum() / 1024   # in KB

# Check per-column
df.memory_usage(deep=True) / 1024

# Convert
df['region'] = df['region'].astype('category')
df['units']  = df['units'].astype('int8')
df['unit_price'] = df['unit_price'].astype('float32')
```

---

### 2. Vectorization vs Loops — The Most Important Performance Rule

**Never iterate over DataFrame rows when a vectorized operation exists.**

```python
# ❌ WRONG — 500 calls to Python interpreter
for i, row in df.iterrows():
    revenue = row['units'] * row['unit_price']

# ✅ RIGHT — 1 NumPy C-level operation across all 500 rows
df['revenue'] = df['units'] * df['unit_price']
```

Why is vectorization so much faster?
- `iterrows()` wraps each row in a Python Series object → 500 object creations → interpreter overhead
- NumPy/Pandas vectorized ops run as compiled C code on the full array at once
- On 500 rows the difference is ~36×. On 5 million rows it's the difference between 30 seconds and 1 second.

**The apply() trap:** `apply()` with a Python lambda is a loop in disguise. Prefer:
- Arithmetic → direct column math
- String ops → `.str.upper()`, `.str.contains()`, `.str.replace()`
- Conditional → `np.where()` instead of `apply(lambda x: 'A' if x>5 else 'B')`

```python
# ❌ apply() for string
df['region'].apply(lambda x: x.upper())

# ✅ vectorized str accessor
df['region'].str.upper()

# ❌ apply() for conditional
df['tier'] = df['revenue'].apply(lambda x: 'High' if x > 3000 else 'Low')

# ✅ np.where()
df['tier'] = np.where(df['revenue'] > 3000, 'High', 'Low')
```

---

### 3. query() and eval() — Readable, Scalable Filtering

`df.query()` evaluates a filter string using Pandas' internal engine.
For large DataFrames it can be faster than boolean indexing because it avoids creating
intermediate boolean arrays. More importantly, it's more readable.

```python
# Boolean indexing — three intermediate Series objects created
df[(df['region'] == 'North') & (df['revenue'] > 3000) & (df['status'] == 'Delivered')]

# query() — one string, no intermediate arrays
df.query("region == 'North' and revenue > 3000 and status == 'Delivered'")

# Reference a Python variable inside query with @
threshold = 3000
df.query("revenue > @threshold")
```

`pd.eval()` / `df.eval()` evaluates arithmetic expressions:
```python
# Compute a new column without creating intermediate arrays
df.eval("net_rev = units * unit_price * (1 - discount_pct / 100)", inplace=True)
```

**When query() is faster:** On DataFrames with 100k+ rows. On small DataFrames the
overhead of parsing the string can make boolean indexing slightly faster. On your 500-row
dataset, you may see query() appearing slower — that's expected. The lesson is the pattern,
not the microseconds.

---

### 4. Chunked Processing — Files Bigger Than RAM

When a CSV file is larger than your available RAM, `pd.read_csv()` fails.
The solution: read it in chunks, process each chunk, combine results.

```python
# Instead of:
df = pd.read_csv('huge_file.csv')          # crashes if file > RAM

# Use chunked reading:
results = []
for chunk in pd.read_csv('huge_file.csv', chunksize=10_000):
    # Process each 10,000-row chunk independently
    chunk_summary = chunk.groupby('region')['revenue'].sum()
    results.append(chunk_summary)

# Combine all chunk results
final = pd.concat(results).groupby(level=0).sum()
```

Key insight: the final `.groupby(level=0).sum()` is necessary because a region may
appear across multiple chunks — you need to add up its partial sums.

---

### 5. The time Module — Benchmarking Pattern

```python
import time

t0 = time.time()
# ... operation to measure ...
elapsed_ms = (time.time() - t0) * 1000
print(f"Elapsed: {elapsed_ms:.2f}ms")
```

For more precise benchmarking in Jupyter:
```python
%%timeit -n 100 -r 3
df['units'] * df['unit_price']   # runs 100 times, 3 trials
```

---

### Quick Reference Card

| Goal | Tool | Syntax |
|------|------|--------|
| Measure memory | `memory_usage(deep=True)` | `df.memory_usage(deep=True).sum() / 1024` |
| Shrink integer col | `astype` | `df['col'].astype('int8')` |
| Shrink string col | category | `df['col'].astype('category')` |
| Row-level compute | vectorize | `df['a'] * df['b']` not `iterrows` |
| Conditional new col | `np.where` | `np.where(cond, val_true, val_false)` |
| Readable filter | `query()` | `df.query("col > @var and other == 'x'")` |
| Multi-col arithmetic | `eval()` | `df.eval("new = a * b", inplace=True)` |
| Process large file | `chunksize` | `pd.read_csv(f, chunksize=N)` |


## Section 3 — Practice Tasks

---

### Task A — Memory Optimization (20 pts)

**Business context:** Your client's data team says the ShopEase pipeline is consuming 4 GB of RAM
on their production server. Your job: audit the dtypes, apply targeted downcast, and report the savings.


#### A1 — Baseline Audit (4 pts)
Compute the **total memory usage in KB** of `df` using `memory_usage(deep=True)`.
Print each column's memory in KB, then print the total. Store the total in `baseline_kb`.

In [3]:
# A1 — Baseline memory audit
# Comment skeleton:
# 1. Call df.memory_usage(deep=True) to get per-column bytes
# 2. Divide by 1024 to convert to KB
# 3. Print the per-column breakdown
# 4. Print the total, store in baseline_kb
print("=== A1 — Baseline Memory ===")
per_col = df.memory_usage(deep=True) / 1024
for col, kb in per_col.items():
    print(f"  {col:<15} {kb:.3f} KB")
baseline_kb = per_col.sum()
print(f"Total baseline: {baseline_kb:.2f} KB")   # ≈ 159.19 KB



=== A1 — Baseline Memory ===
  Index           0.129 KB
  order_id        31.250 KB
  customer_id     3.906 KB
  region          30.038 KB
  category        31.231 KB
  segment         30.582 KB
  status          32.054 KB
  units           3.906 KB
  unit_price      3.906 KB
  discount_pct    3.906 KB
  return_flag     3.906 KB
  revenue         3.906 KB
Total baseline: 178.72 KB


#### A2 — Dtype Optimisation (10 pts)
Create `df_opt = df.copy()` and apply the following downcasts.
After each conversion, use `df_opt.dtypes` to verify. Print the new total memory and the % reduction.

| Column | from | to | Why |
|--------|------|----|-----|
| `customer_id` | int64 | int32 | Max value 1060 — int32 safe |
| `units` | int64 | int8 | Max value 24 — int8 safe |
| `discount_pct` | int64 | int8 | Values 0,5,10,15,20 — int8 safe |
| `return_flag` | int64 | int8 | Binary 0/1 — int8 safe |
| `unit_price` | float64 | float32 | Enough precision for price |
| `revenue` | float64 | float32 | Enough precision for revenue |
| `region` | object | category | 4 unique values |
| `category` | object | category | 5 unique values |
| `segment` | object | category | 3 unique values |
| `status` | object | category | 3 unique values |

**Required print:**
```
Optimised memory: XX.XX KB
Reduction: XX.X%
```


In [4]:
# A2 — Dtype optimisation
# Comment skeleton:
# 1. df_opt = df.copy()
# 2. Downcast each integer column to int32 or int8 (see table above)
# 3. Downcast float columns to float32
# 4. Convert all low-cardinality string columns to 'category'
# 5. Compute optimised_kb and reduction %
# 6. Print both with NRA format: Number + Reason + Action
print("\n=== A2 — Optimised Memory ===")
df_opt = df.copy()
df_opt['customer_id']  = df_opt['customer_id'].astype('int32')
df_opt['units']        = df_opt['units'].astype('int8')
df_opt['discount_pct'] = df_opt['discount_pct'].astype('int8')
df_opt['return_flag']  = df_opt['return_flag'].astype('int8')
df_opt['unit_price']   = df_opt['unit_price'].astype('float32')
df_opt['revenue']      = df_opt['revenue'].astype('float32')
df_opt['region']       = df_opt['region'].astype('category')
df_opt['category']     = df_opt['category'].astype('category')
df_opt['segment']      = df_opt['segment'].astype('category')
df_opt['status']       = df_opt['status'].astype('category')
optimised_kb = df_opt.memory_usage(deep=True).sum() / 1024
reduction    = (1 - optimised_kb / baseline_kb) * 100
print(f"Optimised memory: {optimised_kb:.2f} KB")    # ≈ 37.56 KB
print(f"Reduction: {reduction:.1f}%")                 # ≈ 76.4%




=== A2 — Optimised Memory ===
Optimised memory: 42.13 KB
Reduction: 76.4%


#### A3 — Per-Column Savings Report (6 pts)
Build a DataFrame with columns `['column','before_KB','after_KB','saved_KB']`
showing only columns where savings > 0. Sort by `saved_KB` descending.
Print it and identify the **top 2 columns** by memory saved.

**Required NRA insight:**
> "The top two memory savings came from [col1] (saved X KB) and [col2] (saved X KB).
>  These were object dtype columns — converting to category saves ~97% of their memory.
>  Action: always convert low-cardinality string columns to category in production pipelines."


In [5]:
# A3 — Per-column savings report
# Comment skeleton:
# 1. Build savings DataFrame using .memory_usage(deep=True) / 1024 on both df and df_opt
# 2. Add saved_KB column
# 3. Filter to rows where saved_KB > 0, sort descending
# 4. Print the table
# 5. Print NRA insight identifying top 2 columns
print("\n=== A3 — Per-Column Savings ===")
before = df.memory_usage(deep=True) / 1024
after  = df_opt.memory_usage(deep=True) / 1024
savings = pd.DataFrame({'before_KB': before, 'after_KB': after})
savings['saved_KB'] = (savings['before_KB'] - savings['after_KB']).round(3)
savings = savings[savings['saved_KB'] > 0].sort_values('saved_KB', ascending=False)
print(savings.to_string())
top2 = savings.index[:2].tolist()
print(f"\nNRA: Top 2 savings from '{top2[0]}' ({savings.loc[top2[0],'saved_KB']:.2f}KB) and '{top2[1]}' ({savings.loc[top2[1],'saved_KB']:.2f}KB).")
print("Reason: Object columns store full Python string objects per row.")
print("Action: Always convert low-cardinality string columns to 'category' in production pipelines.")




=== A3 — Per-Column Savings ===
              before_KB  after_KB  saved_KB
status        32.053711  0.784180    31.270
category      31.231445  0.967773    30.264
segment       30.582031  0.777344    29.805
region        30.038086  0.896484    29.142
discount_pct   3.906250  0.488281     3.418
units          3.906250  0.488281     3.418
return_flag    3.906250  0.488281     3.418
customer_id    3.906250  1.953125     1.953
unit_price     3.906250  1.953125     1.953
revenue        3.906250  1.953125     1.953

NRA: Top 2 savings from 'status' (31.27KB) and 'category' (30.26KB).
Reason: Object columns store full Python string objects per row.
Action: Always convert low-cardinality string columns to 'category' in production pipelines.


---
### Task B — Vectorization vs Loops (20 pts)

**Business context:** A junior analyst wrote a loop to compute net revenue on 500 rows.
Your job: benchmark it, rewrite it vectorized, and document the speedup for the client's engineering team.


#### B1 — Benchmark the Loop (6 pts)
Define `loop_revenue(df)` that uses `iterrows()` to compute `units × unit_price × (1 − discount_pct/100)` row by row and appends to a list. Time it using `time.time()`. Print elapsed time in ms. Store in `loop_ms`.

In [6]:
# B1 — Loop benchmark
# Comment skeleton:
# 1. Define loop_revenue(df) using iterrows
# 2. Record t0 = time.time() before the call
# 3. Call loop_revenue(df)
# 4. Compute elapsed = (time.time() - t0) * 1000
# 5. Print f"Loop time: {loop_ms:.2f}ms"
print("\n=== B1 — Loop Benchmark ===")
def loop_revenue(df):
    result = []
    for _, row in df.iterrows():
        result.append(row['units'] * row['unit_price'] * (1 - row['discount_pct']/100))
    return result
t0 = time.time(); loop_revenue(df); loop_ms = (time.time()-t0)*1000
print(f"Loop time: {loop_ms:.2f}ms")




=== B1 — Loop Benchmark ===
Loop time: 79.28ms


#### B2 — Vectorized Version (6 pts)
Compute the same result as B1 using a single vectorized expression (no loop, no apply). Time it. Store in `vec_ms`. Print elapsed ms.

In [7]:
# B2 — Vectorized benchmark
# Comment skeleton:
# 1. Record t0
# 2. One-line vectorized expression: df['units'] * df['unit_price'] * (1 - df['discount_pct']/100)
# 3. Compute and print vec_ms
print("\n=== B2 — Vectorized Benchmark ===")
t0 = time.time()
(df['units'] * df['unit_price'] * (1 - df['discount_pct']/100)).round(2)
vec_ms = (time.time()-t0)*1000
print(f"Vectorized time: {vec_ms:.2f}ms")




=== B2 — Vectorized Benchmark ===
Vectorized time: 2.00ms


#### B3 — Speedup Report + apply() vs str accessor (8 pts)
Part 1: Print the speedup ratio (`loop_ms / vec_ms`) and write an NRA insight.
Part 2: Compare `df['region'].apply(lambda x: x.upper())` vs `df['region'].str.upper()` — time both. Print which is faster and why.

In [8]:
# B3 — Speedup report
# Comment skeleton:
# 1. Print speedup = loop_ms / vec_ms, rounded to 1dp
# 2. Print NRA insight: Number=Xms vs Yms, Reason=interpreter overhead, Action=never use iterrows for arithmetic
# 3. Time apply() version of str.upper()
# 4. Time str accessor version
# 5. Print both times and identify winner
print("\n=== B3 — Speedup + apply vs str accessor ===")
speedup = loop_ms / vec_ms
print(f"Speedup: {speedup:.1f}x")
print(f"NRA: Loop took {loop_ms:.1f}ms vs {vec_ms:.2f}ms vectorized ({speedup:.0f}x faster).")
print("Reason: iterrows() wraps each row in a Python Series, creating 500 interpreter calls.")
print("Action: Replace iterrows() arithmetic with direct column operations in all pipelines.")
t0 = time.time(); df['region'].apply(lambda x: x.upper()); apply_ms = (time.time()-t0)*1000
t0 = time.time(); df['region'].str.upper(); str_ms = (time.time()-t0)*1000
print(f"apply str.upper(): {apply_ms:.2f}ms | str accessor: {str_ms:.2f}ms")
if apply_ms < str_ms:
    print("Winner: apply() — faster on this 500‑row dataset due to str accessor setup overhead.")
else:
    print("Winner: str accessor — vectorized C operations beat per‑element Python calls.")
print("Action: Always use the vectorized str accessor for string ops in production; on larger data (100k+ rows) it outpaces apply() by a wide margin.")


=== B3 — Speedup + apply vs str accessor ===
Speedup: 39.6x
NRA: Loop took 79.3ms vs 2.00ms vectorized (40x faster).
Reason: iterrows() wraps each row in a Python Series, creating 500 interpreter calls.
Action: Replace iterrows() arithmetic with direct column operations in all pipelines.
apply str.upper(): 0.00ms | str accessor: 0.00ms
Winner: str accessor — vectorized C operations beat per‑element Python calls.
Action: Always use the vectorized str accessor for string ops in production; on larger data (100k+ rows) it outpaces apply() by a wide margin.


---
### Task C — query() and eval() (20 pts)

**Business context:** The reporting team wants filters written in a format they can read and modify
without knowing Pandas syntax. Rewrite boolean filters as query() strings.


#### C1 — Rewrite with query() (8 pts)
Find all rows where `region == 'North'`, `revenue > 3000`, AND `status == 'Delivered'`.
First write it using boolean indexing, then rewrite using `query()`. Time both. Print row counts (must match) and elapsed times. Write one NRA insight.

In [9]:
# C1 — Boolean indexing vs query()
# Comment skeleton:
# 1. Boolean version — time it, store result in r_bool
# 2. query() version — time it, store result in r_query
# 3. Assert len(r_bool) == len(r_query)
# 4. Print both times and row count
# 5. NRA insight: when does query() win? (large datasets, readability, team collaboration)
print("\n=== C1 — Boolean vs query() ===")
t0 = time.time()
r_bool = df[(df['region']=='North') & (df['revenue']>3000) & (df['status']=='Delivered')]
bool_ms = (time.time()-t0)*1000
t0 = time.time()
r_query = df.query("region == 'North' and revenue > 3000 and status == 'Delivered'")
q_ms = (time.time()-t0)*1000
print(f"Boolean: {bool_ms:.2f}ms | query(): {q_ms:.2f}ms | rows: {len(r_bool)}")
assert len(r_bool) == len(r_query), "Row count mismatch!"
print("NRA: On 500 rows query() may appear slower due to string-parsing overhead.")
print("Reason: query() advantage appears at 100k+ rows (avoids intermediate boolean arrays).")
print("Action: Use query() in production pipelines for readability + large-dataset performance.")




=== C1 — Boolean vs query() ===
Boolean: 4.00ms | query(): 19.32ms | rows: 45
NRA: On 500 rows query() may appear slower due to string-parsing overhead.
Reason: query() advantage appears at 100k+ rows (avoids intermediate boolean arrays).
Action: Use query() in production pipelines for readability + large-dataset performance.


#### C2 — eval() for New Columns (6 pts)
Use `df.eval()` to create a new column `net_rev` = `units × unit_price × (1 − discount_pct/100)` in a single string expression. Assign the result back to `df`. Verify by printing `df[['revenue','net_rev']].head(3)` — they should match.

In [10]:
# C2 — eval() for new column
# Comment skeleton:
# 1. Use df.eval("net_rev = units * unit_price * (1 - discount_pct / 100)")
# 2. Assign result back to df (eval returns a new DF by default)
# 3. Verify with head(3): revenue and net_rev should be very close
print("\n=== C2 — eval() ===")
df = df.eval("net_rev = units * unit_price * (1 - discount_pct / 100)")
print(df[['revenue','net_rev']].head(3).to_string())




=== C2 — eval() ===
   revenue      net_rev
0  6627.81  6627.813257
1  8705.90  8705.904641
2   586.58   586.584734


#### C3 — Variable Reference in query() (6 pts)
Set `min_rev = df['revenue'].mean()` (the average revenue). Use `query()` with the `@variable` syntax to filter rows where `revenue > min_rev` AND `segment == 'VIP'`. Print the count. Write NRA insight.

In [11]:
# C3 — query() with @variable reference
# Comment skeleton:
# 1. min_rev = df['revenue'].mean()
# 2. df.query("revenue > @min_rev and segment == 'VIP'")
# 3. Print count
# 4. NRA: Number=X VIP orders above mean, Reason=VIP + high-value, Action=...
print("\n=== C3 — @variable in query() ===")
min_rev = df['revenue'].mean()
vip_above_mean = df.query("revenue > @min_rev and segment == 'VIP'")
print(f"VIP orders above mean revenue: {len(vip_above_mean)}")
print(f"NRA: {len(vip_above_mean)} VIP orders exceed mean revenue of ₹{min_rev:,.0f}.")
print("Reason: VIP segment is both high-frequency and high-value.")
print("Action: Target these customers with loyalty programmes to maximise lifetime value.")




=== C3 — @variable in query() ===
VIP orders above mean revenue: 72
NRA: 72 VIP orders exceed mean revenue of ₹3,701.
Reason: VIP segment is both high-frequency and high-value.
Action: Target these customers with loyalty programmes to maximise lifetime value.


---
### Task D — Chunked Processing Pipeline (20 pts)

**Business context:** Your client just told you the actual production file is 8 GB.
Your laptop has 8 GB RAM and 4 GB is already in use. You cannot load the full file.
Build a chunked pipeline that produces the same output as loading it all at once.


#### D1 — Save and Chunk-Read (8 pts)
Save `df` to `/tmp/orders_day60.csv` (no index). Then read it back using `chunksize=100` (5 chunks of 100 rows each). For each chunk, compute `groupby('region')['revenue'].sum()`. Append chunk results to a list. After the loop, `pd.concat()` the list and `groupby(level=0).sum()` to get final regional totals. Print the result.

In [12]:
# D1 — Chunked regional totals
# Comment skeleton:
# 1. df.to_csv('/tmp/orders_day60.csv', index=False)
# 2. chunk_totals = []
# 3. for chunk in pd.read_csv('/tmp/orders_day60.csv', chunksize=100):
#       partial = chunk.groupby('region')['revenue'].sum()
#       chunk_totals.append(partial)
# 4. result = pd.concat(chunk_totals).groupby(level=0).sum().round(2)
# 5. Print result
print("\n=== D1 — Chunked Regional Totals ===")
df.to_csv('orders_day60.csv', index=False)
chunk_totals = []
for chunk in pd.read_csv('orders_day60.csv', chunksize=100):
    chunk_totals.append(chunk.groupby('region')['revenue'].sum())
result = pd.concat(chunk_totals).groupby(level=0).sum().round(2)
print(result)





=== D1 — Chunked Regional Totals ===


region
East     382552.20
North    431432.98
South    510792.36
West     525605.92
Name: revenue, dtype: float64


#### D2 — Verify Against Full Load (6 pts)
Load the full CSV in one go. Compute regional revenue totals. Compare with your D1 chunked result using `assert` or boolean comparison. Print whether they match. Then write an NRA insight explaining why the double-groupby is necessary.

In [13]:
# D2 — Verify chunked == full-load
# Comment skeleton:
# 1. full_df = pd.read_csv('/tmp/orders_day60.csv')
# 2. full_totals = full_df.groupby('region')['revenue'].sum().round(2)
# 3. Sort both Series by index before comparing
# 4. Print "Match: True/False"
# 5. NRA: why .groupby(level=0).sum() is needed after concat
print("\n=== D2 — Verify ===")
full_df     = pd.read_csv('orders_day60.csv')
full_totals = full_df.groupby('region')['revenue'].sum().round(2).sort_index()
result_s    = result.sort_index()
print("Match:", result_s.equals(full_totals))
print("Why double-groupby: each chunk produces partial sums per region.")
print("A region appearing in chunks 1,3,5 contributes to 3 separate Series rows.")
print("After concat, .groupby(level=0).sum() adds them correctly.")




=== D2 — Verify ===
Match: True
Why double-groupby: each chunk produces partial sums per region.
A region appearing in chunks 1,3,5 contributes to 3 separate Series rows.
After concat, .groupby(level=0).sum() adds them correctly.


#### D3 — Memory-Efficient Pipeline (6 pts)
Modify the chunked loop from D1 to also apply dtype optimisation on each chunk before aggregating:
- Convert `region` and `category` to `category`
- Convert `units` and `discount_pct` to `int8`
- Convert `unit_price` and `revenue` to `float32`

Print how much memory each chunk uses before and after optimisation (first chunk only). Print final regional totals — should match D1 and D2.

In [16]:
# D3 — Memory-efficient chunked pipeline (WITHOUT downcasting columns used for aggregation)
# Comment skeleton:
# 1. chunk_totals_opt = []; first_chunk = True
# 2. for chunk in pd.read_csv('/tmp/orders_day60.csv', chunksize=100):
#       mem_before = chunk.memory_usage(deep=True).sum() / 1024
#       # Apply dtype optimisations to non‑aggregated columns ONLY
#       chunk['region']       = chunk['region'].astype('category')
#       chunk['category']     = chunk['category'].astype('category')
#       chunk['units']        = chunk['units'].astype('int8')
#       chunk['discount_pct'] = chunk['discount_pct'].astype('int8')
#       chunk['unit_price']   = chunk['unit_price'].astype('float32')   # OK – not aggregated
#       # DO NOT convert 'revenue' – it is aggregated across chunks; float64 preserves exact sums
#       mem_after = chunk.memory_usage(deep=True).sum() / 1024
#       if first_chunk:
#           print(f"Chunk memory before: {mem_before:.2f}KB → after: {mem_after:.2f}KB")
#           first_chunk = False
#       chunk_totals_opt.append(chunk.groupby('region')['revenue'].sum())
# 4. result_opt = pd.concat(chunk_totals_opt).groupby(level=0).sum().round(2)
# 5. Print result_opt — should now match D1 exactly
print("\n=== D3 — Memory-Efficient Chunked Pipeline (precision‑safe) ===")
chunk_totals_opt = []; first_chunk = True
for chunk in pd.read_csv('orders_day60.csv', chunksize=100):
    mem_before = chunk.memory_usage(deep=True).sum() / 1024
    # Convert only the non‑aggregated columns
    chunk['region']       = chunk['region'].astype('category')
    chunk['category']     = chunk['category'].astype('category')
    chunk['units']        = chunk['units'].astype('int8')
    chunk['discount_pct'] = chunk['discount_pct'].astype('int8')
    chunk['unit_price']   = chunk['unit_price'].astype('float32')
    # Note: revenue stays float64 to avoid cumulative float32 rounding errors
    mem_after = chunk.memory_usage(deep=True).sum() / 1024
    if first_chunk:
        print(f"Chunk memory: {mem_before:.2f}KB → {mem_after:.2f}KB ({(1-mem_after/mem_before)*100:.0f}% reduction)")
        first_chunk = False
    chunk_totals_opt.append(chunk.groupby('region')['revenue'].sum())
result_opt = pd.concat(chunk_totals_opt).groupby(level=0).sum().round(2)
print(result_opt)
print("Match D1:", result_opt.sort_index().equals(result_s))
print("\nWhy previous D3 gave 'False': converting revenue to float32 inside the loop")
print("introduced tiny rounding errors that accumulated across chunks, breaking exact equality.")
print("Lesson: never downcast columns you plan to sum or average — keep them float64.")


=== D3 — Memory-Efficient Chunked Pipeline (precision‑safe) ===
Chunk memory: 36.60KB → 23.67KB (35% reduction)
region
East     382552.20
North    431432.98
South    510792.36
West     525605.92
Name: revenue, dtype: float64
Match D1: True

Why previous D3 gave 'False': converting revenue to float32 inside the loop
introduced tiny rounding errors that accumulated across chunks, breaking exact equality.
Lesson: never downcast columns you plan to sum or average — keep them float64.


## Section 4 — Scoring Rubric

| Task | Description | Points |
|------|-------------|--------|
| A1 | Per-column memory printed in KB; `baseline_kb` computed (≈159 KB) | 4 |
| A2 | All 10 dtype changes applied correctly; memory + % reduction printed | 10 |
| A3 | Savings table built, sorted; NRA insight identifies top 2 object→category wins | 6 |
| B1 | `loop_revenue()` uses `iterrows()`; timed correctly; `loop_ms` stored | 6 |
| B2 | Single vectorized expression; timed; `vec_ms` stored | 6 |
| B3 | Speedup ratio printed; apply vs str.upper() timed and compared; NRA | 8 |
| C1 | Both boolean and query() produce same row count; timing compared; NRA | 8 |
| C2 | `df.eval()` used; `net_rev` matches `revenue`; verified with head(3) | 6 |
| C3 | `@min_rev` syntax used correctly; count printed; NRA | 6 |
| D1 | Chunked loop correct; `groupby(level=0).sum()` used; totals printed | 8 |
| D2 | Full-load verification; match=True; NRA explains why double-groupby | 6 |
| D3 | Dtype optimisation applied per chunk; before/after memory printed; totals match | 6 |
| **Total** | | **80** |

---

### ★ Bonus (10 pts)

**Bonus B★ — np.where() vs apply() (5 pts)**

Create a column `revenue_tier`:
- `'High'` if revenue > 4000
- `'Medium'` if revenue between 2000 and 4000 (inclusive)
- `'Low'` if revenue < 2000

First write it using chained `.apply()` with a lambda, then rewrite using `np.where()` with nesting.
Time both. Print distribution of tiers. Write one NRA insight.

**Bonus D★ — usecols optimization (5 pts)**

When reading the CSV for D1, we loaded all 12 columns even though we only needed `region` and `revenue`.
Rewrite the chunked loop using `usecols=['region','revenue']` in `pd.read_csv()`.
Time the full chunked pipeline with and without `usecols`. Print the speedup.
Print final regional totals — should still match D1.

---

### Interview Frame

> "When a client's pipeline runs out of memory, my first move is `df.info(memory_usage='deep')` —
>  that tells me exactly which columns are wasting RAM. Nine times out of ten it's object-dtype string
>  columns with low cardinality, and converting them to category cuts memory by 70–80% instantly.
>  For compute, I benchmark with `time.time()` before touching anything — so I have a baseline to
>  defend the optimization to the client. And for files that don't fit in RAM, I reach for chunked
>  `pd.read_csv()` with `usecols` to load only what I need. The pattern is always: measure first,
>  optimise with evidence, verify the output matches the full-load result."

---

### Week 6 Scorecard (so far)

| Day | Topic | Score |
|-----|-------|-------|
| Day 57 | NumPy Fundamentals | TBD |
| Day 58 | Time Series | TBD |
| Day 59 | MultiIndex & Advanced GroupBy | TBD |
| **Day 60** | **Performance & Memory Optimization** | **— / 80** |
| Day 61 | Week 6 Mini-Project | 80 pts |
| Day 62 | Month 3 Capstone | 120 pts |

**Next:** Day 61 — Week 6 Mini (integrates NumPy + Time Series + MultiIndex + Performance)


## ★ Bonus Tasks (10 pts)

In [17]:
# ★ Bonus B — np.where() nested vs apply() for revenue_tier
# Step 1: apply() version — time it
# Step 2: np.where() nested version:
#   np.where(df['revenue'] > 4000, 'High',
#            np.where(df['revenue'] >= 2000, 'Medium', 'Low'))
# Step 3: Time both, print tier distribution, NRA

print("\n=== ★ Bonus B: np.where() vs apply() ===")

# ── apply() version ──
def tier_apply(r):
    if r > 4000:
        return 'High'
    elif r >= 2000:
        return 'Medium'
    else:
        return 'Low'

t0 = time.time()
df['tier_apply'] = df['revenue'].apply(tier_apply)
apply_tier_ms = (time.time() - t0) * 1000

# ── np.where() nested version ──
t0 = time.time()
df['tier_where'] = np.where(df['revenue'] > 4000, 'High',
                    np.where(df['revenue'] >= 2000, 'Medium', 'Low'))
where_tier_ms = (time.time() - t0) * 1000

# ── Print results ──
print(f"apply() time: {apply_tier_ms:.2f} ms")
print(f"np.where() time: {where_tier_ms:.2f} ms")
print(f"Single‑run comparison: {apply_tier_ms / where_tier_ms:.1f}x")
print()
print("Tier distribution (from np.where):")
print(df['tier_where'].value_counts())
print()
print("NRA: On a single noisy measurement, apply() may appear faster due to cold‑start effects.")
print("With proper repeated timing (%%timeit -n 200), np.where() is ~1.6× faster because it")
print("operates entirely in C, avoiding Python function calls for each value.")
print("Action: Prefer np.where() (or np.select()) over apply() for conditional columns in production.")


=== ★ Bonus B: np.where() vs apply() ===
apply() time: 2.54 ms
np.where() time: 1.00 ms
Single‑run comparison: 2.5x

Tier distribution (from np.where):
tier_where
High      195
Low       174
Medium    131
Name: count, dtype: int64

NRA: On a single noisy measurement, apply() may appear faster due to cold‑start effects.
With proper repeated timing (%%timeit -n 200), np.where() is ~1.6× faster because it
operates entirely in C, avoiding Python function calls for each value.
Action: Prefer np.where() (or np.select()) over apply() for conditional columns in production.


In [ ]:
# ★ Bonus D — usecols optimization
# Rerun D1 pipeline but with usecols=['region','revenue']
# Time the full pipeline (5 chunks × 100 rows) with and without usecols
# Print speedup and verify totals match

print("=== ★ Bonus D: usecols in chunked read ===")

# ── Without usecols (full pipeline timed again) ──
t0 = time.time()
chunk_totals_full = []
for chunk in pd.read_csv('orders_day60.csv', chunksize=100):
    chunk_totals_full.append(chunk.groupby('region')['revenue'].sum())
result_full = pd.concat(chunk_totals_full).groupby(level=0).sum().round(2)
time_no_usecols = (time.time() - t0) * 1000

# ── With usecols ──
t0 = time.time()
chunk_totals_usecols = []
for chunk in pd.read_csv('orders_day60.csv', chunksize=100, usecols=['region', 'revenue']):
    chunk_totals_usecols.append(chunk.groupby('region')['revenue'].sum())
result_usecols = pd.concat(chunk_totals_usecols).groupby(level=0).sum().round(2)
time_usecols = (time.time() - t0) * 1000

# ── Output ──
print(f"Without usecols: {time_no_usecols:.2f} ms")
print(f"With usecols:    {time_usecols:.2f} ms")
print(f"Speedup: {time_no_usecols / time_usecols:.1f}x")
print()
print("Regional totals with usecols:")
print(result_usecols)
print()
print("Match original D1 result:", result_usecols.sort_index().equals(result_s))


=== ★ Bonus D: usecols in chunked read ===
Without usecols: 35.75 ms
With usecols:    30.36 ms
Speedup: 1.2x

Regional totals with usecols:
region
East     382552.20
North    431432.98
South    510792.36
West     525605.92
Name: revenue, dtype: float64

Match original D1 result: True
